In [2]:
#!pip install librosa
import librosa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

In [3]:
#function used to vectorize single usv snippet from a preprocessed full length recording
# path = file location of the usv snippet, should be .wav
# fs = sampling rate


def usv2vec(path, fs=None,n_fft=4096, hop_length=1024,n_mfcc=13, n_mels=40,fmin=20000, fmax=100000):
    
    
    #print err for bad filepath
    if not Path(path).is_file():
        raise FileNotFoundError(f"Not a file: {path}")
    
    #load usv snippet as np arr (signal) at given fs (sr), use sr = None to use original fs of loaded file. input val to set fs
    signal, sr = librosa.load(path, sr=fs)
    
    #GET THE VECTOR FEATURES%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
    # use same parameters across all methods
    
    #Time domain features (2 total)
    
    #average amp
    rms = np.mean(librosa.feature.rms(y=signal, frame_length=n_fft, hop_length=hop_length))
    #oscillation rate
    zcr = np.mean(librosa.feature.zero_crossing_rate(y=signal, frame_length=n_fft, hop_length=hop_length)) 
    
    
    #Spectral shape features (11 total)
    #mean freq
    centroid  = np.mean(librosa.feature.spectral_centroid (y=signal, sr=sr, n_fft=n_fft, hop_length=hop_length))
    
    #bandwidth (mean freq spread)
    bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=signal, sr=sr, n_fft=n_fft, hop_length=hop_length))
    
    # are there freq peaks or is it white noise (flat across all freqs)
    flatness  = np.mean(librosa.feature.spectral_flatness (y=signal, n_fft=n_fft, hop_length=hop_length))
    
    # mean freq where 85% of energy is below, to change use roll_percent=x (default .85)
    rolloff   = np.mean(librosa.feature.spectral_rolloff (y=signal, sr=sr, n_fft=n_fft, hop_length=hop_length))
    
    #freq amplitude contrast between freq bins (default of 7), to change n_bands=x
    # but may need to adjust freq resolution, n_fft=x
    contrast  = np.mean(librosa.feature.spectral_contrast(y=signal, sr=sr, n_fft=n_fft, hop_length=hop_length,
                                                          n_bands=6), axis=1)
    
    
    
    
    # Pitch features (1 total) 
    # find fundamental freq at each fram
    f0_track = librosa.yin(signal, fmin=fmin, fmax=fmax, sr=sr,frame_length=n_fft, hop_length=hop_length)
    #get average fundamental freq
    f0 = np.nanmean(f0_track)
    #if all frames are silent, set pitch to 0 so Nan error is avoided
    if np.isnan(f0):
        f0 = 0.0
    

    #spectral Contour features (20 total)

    # set number of windows to divide usv snippet in to
    n_contour_points=10
    #(create spectrogram) translate signal to freq domain with short fourier transform, separate magnitude from phase
    S = np.abs(librosa.stft(signal,n_fft=n_fft,hop_length=hop_length))
    
    #create corresponding row of frequency values to use with the spectral infomation
    freqs = librosa.fft_frequencies(sr=sr,n_fft=n_fft)

    # create mask that will be used to strip spectral df of any freq values outside the range of interest
    freq_mask = (freqs >= fmin) & (freqs <= fmax)

    # strip the Sspectrogram df and leave only values between fmin and fmax (usv data)
    S_usv = S[freq_mask, :]
    # strip the  corresponding freq data row df and leave only values between fmin and fmax (usv data)
    freqs_usv = freqs[freq_mask]

    #if we have at least 1 freq bin and at least 2 time points
    if S_usv.shape[0] > 0 and S_usv.shape[1] > 1:
        #find the row with the max power at each time point of the specgtrogram
          #then create a new list that has the freq of max power for each time point
        peak_freq_track = freqs_usv[np.argmax(S_usv, axis=0)]

        #create normalized vector that is the length (duration in time) of the usv
        x_old = np.linspace(0, 1, len(peak_freq_track))
        #create a normalized vector that is the length you want the freq contour data to be
        x_new = np.linspace(0, 1, n_contour_points)

        #resize usv contour to be sure every usv is divided in to n_contourpoint parts 
        #take the peak freq vector, find the freq at each of n_contourpoint steps
          # if the usv is larger than 10 bins this will downsample
          #if the usv is smaller than 10 bins it will interpolate the missing data (stretching the countour)
          #contour_freq is a vector of freq that has the max power at n_contourpoint diffenet evenly divided timepoints of the usv
        contour_freq = np.interp(x_new,x_old,peak_freq_track)
        #find slope of usv at each time step of the freq to get contour shape
        contour_shape = np.gradient(contour_freq)
    #if the usv is too short or has no data, set these vectors to 0
    else:
        contour_freq = np.zeros(n_contour_points)
        contour_shape = np.zeros(n_contour_points)
    
    
    vec = np.concatenate((
        np.array([rms,zcr,centroid,bandwidth,flatness,rolloff,f0], dtype=float),
        contrast.astype(float),
        contour_freq.astype(float),
        contour_shape.astype(float)
        ))
    #returns np arr of 34 features
    return vec

Librosa Functions used for usv2vec:

## Time Domain
rms = average amplitude of the usv (computes root mean squared of overlapping window to create a time series. time series is then averaged)
zcr = average oscillation rate of the usv (times the siganl crosses zero) computes averaged zero crossing of overlapping windows

## Frequency Domian
centroid = what freq is the avergae energy? = average central frequency, split into frames, fourier decomp, computes weighted average of all freq, giving central energy
badnwidth= = how far across freq is power sprread average std of the usv spectrum
flatness= is this a central tone? or noise across all freqs averaged flatness
rolloff= = what is the upper bound of most of the power? mean freq where 85% of energy is below, to change use roll_percent=x (default .85)

contrast= where is there structure in the freq domain = differences of energy in freq bins= split freq into x bands, for each band find highest and lowest energy freq. for each band, peak - valley. 

contour_freq= Take the usv snippet> divide into 10 time windows> find the freq value with the largest power value in each window> this gives the frequency values as the usv changes over time

contour_shape=Take the usv snippet> divide into 10 time windows> find the gradient of the freq in relation to the previos frq> this gives the frequency direction changes over time

## Pitch features
f0= get pitch for each frame, average across time = fundamental freq (perceived pitch)